In [ ]:
from pathlib import Path
import re
import pandas as pd
import xarray as xr

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "scripts").exists() and (candidate / "notebooks").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError(
        "Could not locate the repository root. Start Jupyter from this repository or one of its subdirectories."
    )

REPO_ROOT = find_repo_root(Path.cwd())
RAW_NCAR_DIR = REPO_ROOT / "data" / "raw" / "ncar"
RAW_OISST_DIR = REPO_ROOT / "data" / "raw" / "oisst" / "v2.1"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

root = RAW_OISST_DIR

if not root.exists():
    raise FileNotFoundError(f"The expected OISST root directory was not found: {root}")

month_dirs = sorted(
    p for p in root.iterdir()
    if p.is_dir() and re.fullmatch(r"\d{6}", p.name) and 199101 <= int(p.name) <= 202312
)

files = []
for d in month_dirs:
    files.extend(sorted(d.glob("*.nc")))

if not files:
    raise FileNotFoundError(f"No OISST NetCDF files were found under {root}")

output_file = PROCESSED_DIR / "oisst_jja_1991_2023.nc"

ds = xr.open_mfdataset(
    files,
    combine="nested",
    concat_dim="time",
    coords="minimal",
    parallel=False,
    chunks="auto",
    compat="override",
    combine_attrs="override"
)

ds = ds[["anom"]].sel(time=slice("1991-01-01", "2023-12-31")).sortby("time")

time_index = pd.Index(ds.time.values)
ds = ds.isel(time=~time_index.duplicated())

# JJA only
ds_jja = ds.sel(time=ds.time.dt.month.isin([6, 7, 8]))

ds_jja.to_netcdf(output_file)
